# Train the IngExuity LoRA weights

This notebook fine-tunes `meta-llama/Llama-3.2-1B-Instruct` with 4-bit QLoRA and saves a PEFT adapter compatible with IngExuity's existing `my_weights` artifact.

**Expected environment:** one NVIDIA GPU (Kaggle P100/T4 or a Colab T4-class GPU), Internet access, and JSON/JSONL training data. `HF_TOKEN` is optional for the default public model mirror.

Supported examples:

```json
{"text": "A complete training example"}
{"content": "The text key may also be named content"}
{"messages": [{"role": "user", "content": "I'm overwhelmed."}, {"role": "assistant", "content": "I'm here. Tell me what feels heaviest."}]}
```

The notebook deliberately saves into a fresh run directory. Review the metrics and smoke-test output before replacing `models/trained_model/notebooks/my_weights` in the repository.

## Setup

### 1. Install the training stack

The version ranges stay compatible with this repository's Python requirements while avoiding an unreviewed major-version upgrade.

In [ ]:
import subprocess
import sys

# Kaggle may assign a Pascal P100; its current default torch build omits sm_60.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1", "torchvision==0.20.1",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])

packages = [
    "transformers>=4.50,<5",
    "peft>=0.13,<1",
    "accelerate>=0.34,<2",
    "bitsandbytes>=0.44",
    "datasets>=2.20,<5",
    "huggingface_hub>=0.27,<2",
    "safetensors>=0.4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed. Restart the runtime now only if the notebook environment asks you to.")

## Steps

### 2. Import libraries and record versions

In [ ]:
import json
import math
import os
import platform
import random
import requests
import shutil
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import accelerate
import bitsandbytes
import datasets
import peft
import torch
import transformers
from datasets import Dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "datasets": datasets.__version__,
}
versions

### 3. Configure the run

Set `DATA_PATH` to a JSON or JSONL file. If left blank, the notebook looks for a single `train.jsonl`/`train.json` in the current directory or under `/kaggle/input`.

Effective batch size is `batch_size × gradient_accumulation_steps` (default: 32 sequences). Reduce `max_seq_length` or `batch_size` first if GPU memory is tight.

In [ ]:
IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = "google.colab" in sys.modules
DEFAULT_OUTPUT_ROOT = (
    Path("/kaggle/working/ingexuity_training")
    if IS_KAGGLE
    else Path("/content/ingexuity_training")
    if IS_COLAB
    else Path.cwd() / "artifacts" / "ingexuity_training"
)

@dataclass
class RunConfig:
    model_name: str = "unsloth/Llama-3.2-1B-Instruct"
    data_path: str = ""  # Example: /content/train.jsonl
    eval_data_path: str = ""  # Optional held-out family split
    output_root: str = str(DEFAULT_OUTPUT_ROOT)
    seed: int = 42
    eval_fraction: float = 0.10
    max_seq_length: int = 512
    num_epochs: float = 3.0
    batch_size: int = 4
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    lora_rank: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    logging_steps: int = 10
    run_initial_eval: bool = True
    resume_from_checkpoint: bool = True

config = RunConfig()
OUTPUT_ROOT = Path(config.output_root)
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
FINAL_ADAPTER_DIR = OUTPUT_ROOT / "my_weights"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
set_seed(config.seed)
asdict(config)

### 4. Upload data or pull it from GitHub

Choose one source in the form below:

- `synthetic`: clone the configured IngExuity ref, generate a validated 100-row smoke dataset, and use its family-isolated train/eval splits.
- `upload`: select one local `.json` or `.jsonl` file in Colab.
- `github`: download one file from a public or private GitHub repository through the Contents API. For a private repository, add a fine-grained, read-only personal access token as a Colab secret named `GH_TOKEN` and grant notebook access to that secret. The token is never printed, stored in the notebook, embedded in a URL, or written to the training manifest.
- `path`: use `config.data_path` exactly as configured in the previous cell.

For GitHub mode, set the repository as `owner/repository`, the path relative to the repository root, and a branch, tag, or commit in `GITHUB_REF`.

In [ ]:
DATA_SOURCE = "synthetic"  # @param ["synthetic", "upload", "github", "path"]
SYNTHETIC_SMOKE_COUNT = 100
SYNTHETIC_REPO_URL = "https://github.com/toxzak-svg/ingexuity.git"
SYNTHETIC_REPO_REF = "codex/kaggle-synthetic-pilot"
GITHUB_REPOSITORY = "owner/repository"  # @param {type:"string"}
GITHUB_FILE_PATH = "data/train.jsonl"  # @param {type:"string"}
GITHUB_REF = "main"  # @param {type:"string"}

def get_colab_secret(name):
    if not IS_COLAB:
        return os.environ.get(name)
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def acquire_training_data():
    mode = DATA_SOURCE.strip().lower()
    if mode == "synthetic":
        source_root = next(
            (candidate for candidate in [Path.cwd(), Path("/kaggle/working/ingexuity-source"), Path("/content/ingexuity-source")]
             if (candidate / "python" / "build_synthetic_dataset.py").is_file()),
            None,
        )
        if source_root is None:
            source_root = Path("/kaggle/working/ingexuity-source") if IS_KAGGLE else Path("/content/ingexuity-source")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", SYNTHETIC_REPO_REF, SYNTHETIC_REPO_URL, str(source_root)],
                check=True,
            )
        output_dir = Path("/kaggle/working/synthetic_pilot") if IS_KAGGLE else Path("/content/synthetic_pilot")
        subprocess.run(
            [sys.executable, str(source_root / "python" / "build_synthetic_dataset.py"), "--output", str(output_dir), "--count", str(SYNTHETIC_SMOKE_COUNT), "--seed", str(config.seed)],
            cwd=str(source_root),
            check=True,
        )
        manifest = json.loads((output_dir / "manifest.json").read_text(encoding="utf-8"))
        if manifest["accepted"] != manifest["requested"] or manifest["rejected"]:
            raise RuntimeError(f"Synthetic validation failed: {manifest}")
        if any(manifest["family_overlap"].values()):
            raise RuntimeError(f"Synthetic family leakage detected: {manifest['family_overlap']}")
        config.eval_data_path = str(output_dir / "eval.jsonl")
        config.num_epochs = 1.0
        print(json.dumps(manifest, indent=2, sort_keys=True))
        return output_dir / "train.jsonl"

    if mode == "path":
        if not config.data_path:
            raise ValueError("Set config.data_path in the previous cell when DATA_SOURCE is 'path'.")
        return Path(config.data_path).expanduser()

    if mode == "upload":
        if not IS_COLAB:
            raise RuntimeError("Upload mode requires Colab. Use path mode outside Colab.")
        from google.colab import files
        uploaded = files.upload()
        candidates = [name for name in uploaded if Path(name).suffix.lower() in {".json", ".jsonl"}]
        if len(candidates) != 1:
            raise ValueError(f"Upload exactly one .json or .jsonl file; received: {list(uploaded)}")
        return Path("/content") / candidates[0]

    if mode == "github":
        repository = GITHUB_REPOSITORY.strip().strip("/")
        file_path = GITHUB_FILE_PATH.strip().strip("/")
        ref = GITHUB_REF.strip()
        if repository.count("/") != 1 or not file_path or not ref:
            raise ValueError("Set GITHUB_REPOSITORY='owner/repository', GITHUB_FILE_PATH, and GITHUB_REF.")

        gh_token = get_colab_secret("GH_TOKEN")
        headers = {
            "Accept": "application/vnd.github.raw+json",
            "X-GitHub-Api-Version": "2022-11-28",
        }
        if gh_token:
            headers["Authorization"] = f"Bearer {gh_token}"

        owner, repository_name = repository.split("/", 1)
        api_url = (
            f"https://api.github.com/repos/{quote(owner)}/{quote(repository_name)}"
            f"/contents/{quote(file_path, safe='/')}"
        )
        response = requests.get(api_url, headers=headers, params={"ref": ref}, timeout=60)
        if response.status_code == 404:
            raise RuntimeError(
                "GitHub file not found. Check repository/path/ref and ensure GH_TOKEN can read the repository."
            )
        response.raise_for_status()

        destination = Path("/content") / Path(file_path).name
        if destination.suffix.lower() not in {".json", ".jsonl"}:
            raise ValueError("The GitHub training file must end in .json or .jsonl.")
        destination.write_bytes(response.content)
        print(f"Downloaded {repository}@{ref}:{file_path} -> {destination} ({destination.stat().st_size:,} bytes)")
        return destination

    raise ValueError("DATA_SOURCE must be 'synthetic', 'upload', 'github', or 'path'.")

config.data_path = str(acquire_training_data())
print(f"Training data ready: {config.data_path}")

### 5. Authenticate and verify the GPU

When using a gated model source, accept its license and store a read token as `HF_TOKEN` in Colab Secrets, Kaggle Secrets, or the environment. The default public mirror does not require a token. Tokens are never printed or written into the output manifest.

In [ ]:
def find_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    if IS_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("HF_TOKEN")
        except Exception:
            pass
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    return None

hf_token = find_hf_token()
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("Public model mirror selected; continuing without HF_TOKEN.")

if not torch.cuda.is_available():
    raise RuntimeError("An NVIDIA GPU is required for this QLoRA run. Enable a GPU accelerator and restart.")

gpu = torch.cuda.get_device_properties(0)
gpu_summary = {
    "name": gpu.name,
    "vram_gb": round(gpu.total_memory / 1024**3, 2),
    "cuda_runtime": torch.version.cuda,
    "bf16_supported": torch.cuda.is_bf16_supported(),
}
gpu_summary

### 6. Load and validate the training examples

In [ ]:
def resolve_data_path(configured_path):
    if configured_path:
        candidate = Path(configured_path).expanduser()
        if not candidate.exists():
            raise FileNotFoundError(f"Configured data file does not exist: {candidate}")
        return candidate

    candidates = [Path.cwd() / "train.jsonl", Path.cwd() / "train.json"]
    if IS_KAGGLE:
        candidates.extend(Path("/kaggle/input").rglob("train.jsonl"))
        candidates.extend(Path("/kaggle/input").rglob("train.json"))
    matches = sorted({path.resolve() for path in candidates if path.is_file()})
    if len(matches) != 1:
        raise FileNotFoundError(
            "Set config.data_path to one JSON/JSONL file. Auto-discovery found: "
            + (", ".join(map(str, matches)) if matches else "none")
        )
    return matches[0]

def read_records(path):
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as handle:
            records = [json.loads(line) for line in handle if line.strip()]
    elif path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if isinstance(payload, list):
            records = payload
        elif isinstance(payload, dict):
            records = payload.get("data", payload.get("texts", []))
        else:
            records = []
    else:
        raise ValueError("Training data must use the .json or .jsonl extension.")
    return records

def normalize_record(record, index):
    if isinstance(record, str):
        record = {"text": record}
    if not isinstance(record, dict):
        raise ValueError(f"Example {index} must be a string or object, got {type(record).__name__}.")

    messages = record.get("messages")
    if messages is not None:
        if not isinstance(messages, list) or not messages:
            raise ValueError(f"Example {index} has an empty or invalid messages list.")
        for message in messages:
            if not isinstance(message, dict) or message.get("role") not in {"system", "user", "assistant"}:
                raise ValueError(f"Example {index} contains an invalid chat role.")
            if not isinstance(message.get("content"), str) or not message["content"].strip():
                raise ValueError(f"Example {index} contains an empty chat message.")
        return {"messages": messages}

    text = record.get("text", record.get("content"))
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"Example {index} needs non-empty text/content or messages.")
    return {"text": text.strip()}

def load_unique_records(path):
    normalized = [normalize_record(item, index) for index, item in enumerate(read_records(path))]
    unique_by_content = {}
    for record in normalized:
        key = json.dumps(record, ensure_ascii=False, sort_keys=True)
        unique_by_content.setdefault(key, record)
    return normalized, list(unique_by_content.values())

data_path = resolve_data_path(config.data_path)
normalized, records = load_unique_records(data_path)
if config.eval_data_path:
    eval_data_path = resolve_data_path(config.eval_data_path)
    eval_normalized, eval_records = load_unique_records(eval_data_path)
    train_records = records
    train_keys = {json.dumps(record, ensure_ascii=False, sort_keys=True) for record in train_records}
    eval_keys = {json.dumps(record, ensure_ascii=False, sort_keys=True) for record in eval_records}
    if train_keys & eval_keys:
        raise ValueError("Configured train and eval files contain duplicate examples.")
else:
    if len(records) < 2:
        raise ValueError("At least two valid, unique examples are required for a train/eval split.")
    if not 0 < config.eval_fraction < 0.5:
        raise ValueError("eval_fraction must be greater than 0 and less than 0.5.")
    random.Random(config.seed).shuffle(records)
    eval_count = max(1, round(len(records) * config.eval_fraction))
    eval_records = records[:eval_count]
    train_records = records[eval_count:]
    eval_data_path = None
    eval_normalized = eval_records
if not train_records or not eval_records:
    raise ValueError("Training and evaluation splits must both be non-empty.")
data_summary = {
    "source": str(data_path),
    "eval_source": str(eval_data_path) if eval_data_path else "derived",
    "raw_examples": len(normalized) + (len(eval_normalized) if config.eval_data_path else 0),
    "unique_examples": len(train_records) + len(eval_records),
    "duplicates_removed": (len(normalized) - len(records)) + (len(eval_normalized) - len(eval_records) if config.eval_data_path else 0),
    "train_examples": len(train_records),
    "eval_examples": len(eval_records),
}
data_summary

### 7. Load the tokenizer and render chat examples

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, token=hf_token, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def render_record(record):
    if "messages" in record:
        return tokenizer.apply_chat_template(
            record["messages"], tokenize=False, add_generation_prompt=False
        )
    bos = tokenizer.bos_token or ""
    eos = tokenizer.eos_token or ""
    return bos + record["text"] + eos

train_texts = [render_record(record) for record in train_records]
eval_texts = [render_record(record) for record in eval_records]
print("Rendered preview (truncated):")
print(train_texts[0][:500])

### 8. Tokenize and inspect sequence lengths

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=True,
        max_length=config.max_seq_length,
        padding=False,
    )

train_dataset = Dataset.from_dict({"text": train_texts}).map(
    tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing train split"
)
eval_dataset = Dataset.from_dict({"text": eval_texts}).map(
    tokenize_batch, batched=True, remove_columns=["text"], desc="Tokenizing eval split"
)

train_lengths = [len(ids) for ids in train_dataset["input_ids"]]
length_summary = {
    "minimum": min(train_lengths),
    "median": sorted(train_lengths)[len(train_lengths) // 2],
    "maximum": max(train_lengths),
    "at_length_limit": sum(length == config.max_seq_length for length in train_lengths),
}
length_summary

### 9. Load the 4-bit base model and attach LoRA

In [ ]:
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    token=hf_token,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
lora_config = LoraConfig(
    r=config.lora_rank,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
parameter_summary = {
    "trainable": trainable,
    "total": total,
    "trainable_percent": round(100 * trainable / total, 4),
}
parameter_summary

### 10. Build the trainer

Dynamic padding keeps batches compact. The custom collator sets only padded label positions to `-100`, so padding does not contribute to the causal-language-model loss while real end-of-sequence tokens remain supervised.

In [ ]:
class CausalLMDataCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        batch["labels"] = batch["input_ids"].clone()
        batch["labels"][batch["attention_mask"] == 0] = -100
        return batch

data_collator = CausalLMDataCollator(tokenizer)
sample_batch = data_collator([train_dataset[index] for index in range(min(2, len(train_dataset)))])
padding_mask = sample_batch["attention_mask"] == 0
assert torch.all(sample_batch["labels"][padding_mask] == -100), "Padding labels must be masked."
assert torch.any(sample_batch["labels"] != -100), "The batch has no supervised tokens."

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=False,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=config.weight_decay,
    max_grad_norm=config.max_grad_norm,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=config.logging_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    group_by_length=True,
    report_to="none",
    seed=config.seed,
    data_seed=config.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)
print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")

## Checks

### 11. Measure the starting evaluation loss

In [ ]:
initial_metrics = trainer.evaluate(metric_key_prefix="initial") if config.run_initial_eval else {}
if "initial_loss" in initial_metrics:
    initial_metrics["initial_perplexity"] = math.exp(min(initial_metrics["initial_loss"], 20))
initial_metrics

### 12. Train (or resume)

In [ ]:
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None
resume_checkpoint = last_checkpoint if config.resume_from_checkpoint else None
print(f"Resume checkpoint: {resume_checkpoint or 'none'}")

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
train_metrics = dict(train_result.metrics)
train_metrics

### 13. Evaluate and save the adapter

In [ ]:
final_metrics = trainer.evaluate(metric_key_prefix="final")
if "final_loss" in final_metrics:
    final_metrics["final_perplexity"] = math.exp(min(final_metrics["final_loss"], 20))

FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": config.model_name,
    "adapter_path": str(FINAL_ADAPTER_DIR),
    "config": asdict(config),
    "lora_target_modules": target_modules,
    "data": data_summary,
    "sequence_lengths": length_summary,
    "parameters": parameter_summary,
    "gpu": gpu_summary,
    "versions": versions,
    "initial_metrics": initial_metrics,
    "train_metrics": train_metrics,
    "final_metrics": final_metrics,
}
with (FINAL_ADAPTER_DIR / "training_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

expected_files = ["adapter_config.json", "adapter_model.safetensors", "training_manifest.json"]
missing = [name for name in expected_files if not (FINAL_ADAPTER_DIR / name).is_file()]
assert not missing, f"Adapter save is incomplete; missing: {missing}"
{"saved_to": str(FINAL_ADAPTER_DIR), **final_metrics}

### 14. Run a generation smoke test

In [ ]:
model.config.use_cache = True
model.eval()
smoke_messages = [
    {"role": "system", "content": "You are IngExuity: empathetic, direct, and prediction-first."},
    {"role": "user", "content": "I keep saying I'm fine, but I have not slept and everything feels like too much."},
]
prompt = tokenizer.apply_chat_template(smoke_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
continuation = generated[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(continuation, skip_special_tokens=True).strip())

## Next Steps

### 15. Package the weights

The ZIP contains only the saved adapter/tokenizer/manifest directory, not the gated base model. After reviewing the final metrics and smoke test, replace the repository's `models/trained_model/notebooks/my_weights` directory with the unpacked adapter and run the project's inference smoke test.

In [ ]:
zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT / "ingexuity_my_weights"), "zip", FINAL_ADAPTER_DIR))
print(f"Packaged adapter: {zip_path} ({zip_path.stat().st_size / 1024**2:.2f} MiB)")

if IS_COLAB:
    from google.colab import files
    files.download(str(zip_path))